In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/churn-bigml-20.csv")

Mounted at /content/drive


In [ ]:
df.shape

(667, 20)

In [ ]:
df.isnull().sum()

,0
State,0
Account length,0
Area code,0
International plan,0
Voice mail plan,0
Number vmail messages,0
Total day minutes,0
Total day calls,0
Total day charge,0
Total eve minutes,0


In [ ]:
duplicate_rows = df[df.duplicated()]
print(f"\n No of duplicate rows: {duplicate_rows.shape[0]}")


 No of duplicate rows: 0


In [ ]:
x =df.drop("Churn",axis=1)
y = df["Churn"]

In [ ]:
# encode catagori
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder()
for col in x.select_dtypes(include='object').columns:
  x[col] = le.fit_transform(x[col])

if y.dtype == 'object':
  y = le.fit_transform(y)

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [ ]:
# Train Test
from sklearn.model_selection import train_test_split, GridSearchCV
x_train,x_test,y_train,y_test = train_test_split(x_scaled, y, test_size = 0.25, random_state=42, stratify=y)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# model
models = {
    "LogisticRegression": LogisticRegression(max_iter = 1000),
    "Decision Tree": DecisionTreeClassifier(random_state = 42),
    "Random Forest": RandomForestClassifier(random_state = 42)
}


In [ ]:
results = []

for name, model in models.items():
  model.fit(x_train,y_train)
  y_pred = model.predict(x_test)

  results.append({
      "Model": name,
      "Accuracy": accuracy_score(y_test,y_pred),
      "Precision": precision_score(y_test,y_pred),
      "Recall": recall_score(y_test,y_pred),
      "f1_score": f1_score(y_test,y_pred)
  })

results_df = pd.DataFrame(results)
results_df


,Model,Accuracy,Precision,Recall,f1_score
0,LogisticRegression,0.856287,0.500000,0.125000,0.200000
1,Decision Tree,0.880240,0.600000,0.500000,0.545455
2,Random Forest,0.916168,0.916667,0.458333,0.611111


In [ ]:
best_model = RandomForestClassifier(random_state=42)
best_model.fit(x_train,y_train)
y_pred = best_model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.92      0.99      0.95       143
        True       0.92      0.46      0.61        24

    accuracy                           0.92       167
   macro avg       0.92      0.73      0.78       167
weighted avg       0.92      0.92      0.90       167



In [ ]:
# Gridsearch
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]

}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv = 5,
    scoring = 'f1',
    n_jobs = 1
)
grid.fit(x_train, y_train)
print("Best Parameter: ", grid.best_params_)

Best Parameter:  {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [ ]:
best_rf = grid.best_estimator_
y_pred_tuned = best_rf.predict(x_test)

print("Accuracy", accuracy_score(y_test,y_pred_tuned)),
print("Precision", precision_score(y_test,y_pred_tuned)),
print("Recall", recall_score(y_test,y_pred_tuned)),
print("f1_score", f1_score(y_test,y_pred_tuned))

Accuracy 0.9161676646706587
Precision 0.9166666666666666
Recall 0.4583333333333333
f1_score 0.6111111111111112
